# 팀원 CLEAN 테이블 CSV 재현: AI 전력예측과 생산시간 분산

팀원 `forecast.ipynb`의 제조공정 관련 셀만 추려 로컬 CSV에 적용한다. 지역 전력피크와 제조공정 고부하시간 중첩은 적용 가능성 분석이며, 제조 CSV 자체에는 지역·공장 식별자가 없다.

사용한 입력은 Snowflake `SMU_HACKATHON_DB.CLEAN.MANUFACTURING_POWER_CLEAN`에서 내려받은 CSV export이다.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
DERIVED = DATA / "derived"
SOURCE = DATA / "Untitled 7_2026-08-19-1631.csv"
REGIONAL_SOURCE = DATA / "한국전력거래소_지역별 시간대별 전력거래량_20251231-공공데이터포털.csv"

## 1. 팀원 방식의 전처리와 변수

팀원 코드와 동일하게 `시간`을 0–23시로 제한하고, 모델에는 시간·생산량·기상·공장인원 7개 변수만 사용한다. 원본의 비가동 시간 공장인원 NULL은 모델 실행을 위해 0으로 보완한다.

In [2]:
df = pd.read_csv(SOURCE, encoding="utf-8-sig").rename(columns={
    "날짜": "DATE_ID", "시간": "HOUR", "생산량": "PRODUCTION_QTY",
    "기온": "TEMPERATURE", "풍속": "WIND_SPEED", "습도": "HUMIDITY",
    "강수량": "PRECIPITATION", "공장인원": "FACTORY_WORKERS", "평균": "AVG_POWER",
})
numeric = ["DATE_ID", "HOUR", "PRODUCTION_QTY", "TEMPERATURE", "WIND_SPEED", "HUMIDITY", "PRECIPITATION", "FACTORY_WORKERS", "AVG_POWER"]
for col in numeric:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df[df["HOUR"].between(0, 23)].copy()
df["FACTORY_WORKERS"] = df["FACTORY_WORKERS"].fillna(0)
df["PRODUCTION_QTY"] = df["PRODUCTION_QTY"].astype(float)
df = df.sort_values(["DATE_ID", "HOUR"]).reset_index(drop=True)
features = ["HOUR", "PRODUCTION_QTY", "TEMPERATURE", "WIND_SPEED", "HUMIDITY", "PRECIPITATION", "FACTORY_WORKERS"]
target = "AVG_POWER"
print({"rows": len(df), "dates": df["DATE_ID"].nunique(), "features": features})

{'rows': 6120, 'dates': 255, 'features': ['HOUR', 'PRODUCTION_QTY', 'TEMPERATURE', 'WIND_SPEED', 'HUMIDITY', 'PRECIPITATION', 'FACTORY_WORKERS']}


## 2. 팀원 방식의 Random Forest 예측모델

앞 80% 행을 학습, 뒤 20% 행을 테스트하고 `n_estimators=200`, `max_depth=12`, `min_samples_leaf=3`, `random_state=42`를 사용한다.

In [3]:
split = int(len(df) * 0.8)
X, y = df[features], df[target]
model = RandomForestRegressor(n_estimators=200, max_depth=12, min_samples_leaf=3, random_state=42, n_jobs=-1)
model.fit(X.iloc[:split], y.iloc[:split])
pred = model.predict(X.iloc[split:])
metrics = pd.DataFrame([{
    "TRAIN_ROWS": split, "TEST_ROWS": len(df) - split,
    "MAE": mean_absolute_error(y.iloc[split:], pred),
    "RMSE": np.sqrt(mean_squared_error(y.iloc[split:], pred)),
    "R2": r2_score(y.iloc[split:], pred),
}])
metrics.to_csv(DERIVED / "manufacturing_ai_peak_team_clean_export_model_metrics_2021_v1.csv", index=False, encoding="utf-8-sig")
display(metrics.round(4))

,TRAIN_ROWS,TEST_ROWS,MAE,RMSE,R2
0,4896,1224,12.4631,18.2996,0.9112


## 3. 지역 전력피크와 제조공정 고부하시간 중첩

제조공정 평균 전력사용량 상위 9시간과 2025년 지역별 평균 전력거래량 상위 5시간을 비교한다. 전남은 4/5시간이 겹치는지 검증한다.

In [4]:
regional = pd.read_csv(REGIONAL_SOURCE, encoding="utf-8-sig")
regional["거래시간"] = pd.to_numeric(regional["거래시간"], errors="coerce")
regional["전력거래량"] = pd.to_numeric(regional["전력거래량"], errors="coerce")
regional_hourly = regional.groupby(["지역", "거래시간"], as_index=False)["전력거래량"].mean()
regional_hourly = regional_hourly.rename(columns={"거래시간": "HOUR", "전력거래량": "AVG_POWER_TRADING"})
manufacturing_hourly = df.groupby("HOUR", as_index=False)[target].mean().rename(columns={target: "AVG_POWER_USAGE"})
high_hours = list(manufacturing_hourly.sort_values("AVG_POWER_USAGE", ascending=False).head(9)["HOUR"].astype(int))
overlap_rows = []
for region in ["전라남도", "경상북도", "충청남도"]:
    top = regional_hourly[regional_hourly["지역"] == region].sort_values("AVG_POWER_TRADING", ascending=False).head(5)
    top_hours = list(top["HOUR"].astype(int))
    overlap_hours = sorted(set(top_hours) & set(high_hours))
    overlap_rows.append({"REGION": region, "REGION_TOP5_HOURS": ",".join(map(str, top_hours)), "MANUFACTURING_HIGH_LOAD_HOURS_TOP9": ",".join(map(str, sorted(high_hours))), "OVERLAP_HOURS": ",".join(map(str, overlap_hours)), "OVERLAP_RATE_PCT": len(overlap_hours) / 5 * 100})
overlap = pd.DataFrame(overlap_rows)
overlap.to_csv(DERIVED / "regional_manufacturing_peak_overlap_team_clean_export_2025_v1.csv", index=False, encoding="utf-8-sig")
display(overlap)

,REGION,REGION_TOP5_HOURS,MANUFACTURING_HIGH_LOAD_HOURS_TOP9,OVERLAP_HOURS,OVERLAP_RATE_PCT
0,전라남도,"17,18,15,16,14","8,9,10,11,13,14,15,16,18","14,15,16,18",80.0
1,경상북도,"20,21,19,22,23","8,9,10,11,13,14,15,16,18",,0.0
2,충청남도,"20,19,21,18,22","8,9,10,11,13,14,15,16,18",18,20.0


## 4. 팀원 방식의 생산시간 분산 시뮬레이션

고부하 시간은 위에서 계산한 9개 시간으로 고정한다. 저부하 시간은 날짜별 평균 예측전력의 90% 이하로 자동 선정하고, 이동 생산량은 저부하 시간의 피크 대비 여유도에 비례해 배분한다. 총생산량은 보존한다.

In [5]:
base_df = df.copy()
base_df["BASE_PRED_POWER"] = model.predict(base_df[features])
fixed_high_hours = high_hours
shift_rates = [0.10, 0.20, 0.30, 0.40]

def simulate_one_day(day_df, shift_rate):
    day = day_df.copy()
    before_peak = day["BASE_PRED_POWER"].max()
    before_peak_hour = day.loc[day["BASE_PRED_POWER"].idxmax(), "HOUR"]
    hourly = day.groupby("HOUR")["BASE_PRED_POWER"].mean()
    low_hours = hourly[hourly <= hourly.mean() * 0.90].index.tolist()
    high_mask = day["HOUR"].isin(fixed_high_hours)
    original_high = day.loc[high_mask, "PRODUCTION_QTY"].sum()
    day.loc[high_mask, "PRODUCTION_QTY"] *= (1 - shift_rate)
    removed = original_high - day.loc[high_mask, "PRODUCTION_QTY"].sum()
    headroom = before_peak - hourly.loc[low_hours]
    weights = headroom / headroom.sum()
    for hour in low_hours:
        mask = day["HOUR"] == hour
        day.loc[mask, "PRODUCTION_QTY"] += removed * weights.loc[hour] / mask.sum()
    day["AFTER_PRED_POWER"] = model.predict(day[features])
    after_peak = day["AFTER_PRED_POWER"].max()
    after_peak_hour = day.loc[day["AFTER_PRED_POWER"].idxmax(), "HOUR"]
    return {
        "DATE_ID": day_df["DATE_ID"].iloc[0], "SHIFT_RATE": shift_rate * 100,
        "BEFORE_PEAK": before_peak, "AFTER_PEAK": after_peak,
        "PEAK_REDUCTION": before_peak - after_peak,
        "REDUCTION_RATE_PCT": (before_peak - after_peak) / before_peak * 100,
        "BEFORE_PEAK_HOUR": before_peak_hour, "AFTER_PEAK_HOUR": after_peak_hour,
        "PRODUCTION_DIFF": day["PRODUCTION_QTY"].sum() - day_df["PRODUCTION_QTY"].sum(),
    }

results = [simulate_one_day(day, rate) for _, day in base_df.groupby("DATE_ID") for rate in shift_rates]
daily_results = pd.DataFrame(results)
daily_results.to_csv(DERIVED / "manufacturing_ai_peak_team_clean_export_daily_2021_v1.csv", index=False, encoding="utf-8-sig")
summary_rows = []
for rate, group in daily_results.groupby("SHIFT_RATE"):
    test = wilcoxon(group["BEFORE_PEAK"], group["AFTER_PEAK"], alternative="greater", zero_method="wilcox")
    summary_rows.append({"SHIFT_RATE": rate, "N_DAYS": len(group), "AVG_BEFORE_PEAK": group["BEFORE_PEAK"].mean(), "AVG_AFTER_PEAK": group["AFTER_PEAK"].mean(), "AVG_REDUCTION_RATE_PCT": group["REDUCTION_RATE_PCT"].mean(), "REDUCED_DAYS": int((group["PEAK_REDUCTION"] > 0).sum()), "UNCHANGED_DAYS": int(np.isclose(group["PEAK_REDUCTION"], 0).sum()), "INCREASED_DAYS": int((group["PEAK_REDUCTION"] < 0).sum()), "MAX_PRODUCTION_DIFF": group["PRODUCTION_DIFF"].abs().max(), "WILCOXON_P_VALUE_ONE_SIDED": test.pvalue})
summary = pd.DataFrame(summary_rows)
summary.to_csv(DERIVED / "manufacturing_ai_peak_team_clean_export_summary_2021_v1.csv", index=False, encoding="utf-8-sig")
display(summary.round(4))

,SHIFT_RATE,N_DAYS,AVG_BEFORE_PEAK,AVG_AFTER_PEAK,AVG_REDUCTION_RATE_PCT,REDUCED_DAYS,UNCHANGED_DAYS,INCREASED_DAYS,MAX_PRODUCTION_DIFF,WILCOXON_P_VALUE_ONE_SIDED
0,10.0,255,137.4081,136.8867,0.3019,192,104,29,0.0,0.0
1,20.0,255,137.4081,136.3625,0.5894,190,101,38,0.0,0.0
2,30.0,255,137.4081,135.9966,0.7106,185,98,34,0.0,0.0
3,40.0,255,137.4081,135.3350,1.0616,188,98,40,0.0,0.0


## 해석상 주의

이 Notebook은 팀원 코드의 핵심 로직을 로컬 CSV에 옮긴 재현본이다. 팀원의 Snowflake 테이블 생성 시점, NULL 처리, 원본 값 상태가 달라 R²와 피크 수치가 완전히 일치하지 않을 수 있다. 반면 고부하시간·저부하시간 선정, 생산량 보존, 여유도 배분, 날짜별 피크 및 Wilcoxon 방식은 동일하게 맞췄다. 제조 CSV에 지역 식별자가 없으므로 결과는 전남 실제 감축량이 아니라 대표 공정의 적용 가능성 사례다.